# I-E04 — I-N120 vs E03 Rescue / Miss / FP Analysis

N120 단독의 F2 우선 threshold `0.15`와 E03 ensemble의 선택 threshold `0.20`을 비교한다.

- **e03_rescue**: N120이 미탐했지만 E03가 탐지한 GT 병변
- **e03_regression**: N120은 탐지했지만 E03가 미탐한 GT 병변
- **missed_by_both**: 두 후보가 모두 미탐한 GT 병변
- FP는 GT와 겹치지 않는 예측 connected component 기준

이 분석은 동일 validation set에서 후보를 설명하기 위한 것이며, 최종 성능 확정은 독립 test/hold-out에서 해야 한다.


In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
LABEL_DIR = Path("/home/stu03/workspace/datasets/ischemic/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations")
N120_PROB = REPO / "runs/I-nnunet/Dataset502_ISLES2022/nnUNetTrainerIschemiaMultimodal__IschemiaPlansDualEncoderGatedV1__3d_fullres/fold_0/best_probability_validation"
E03_PRED = REPO / "runs/ischemia_ensemble/I-E03_N120_S50_S101/N120_060_S101_020_S50_020/probabilities"
OUT = REPO / "runs/ischemia_ensemble/I-E04_N120_vs_E03_rescue"
N120_THRESHOLD, E03_THRESHOLD = .15, .20
assert all(x.exists() for x in [PYTHON, LABEL_DIR, N120_PROB, E03_PRED])


## Rescue 분석 실행


In [ ]:
command = [
    str(PYTHON), str(REPO / "scripts/ischemia/ensemble/analyze_n120_e03_rescue.py"),
    "--n120-probabilities", str(N120_PROB),
    "--e03-predictions", str(E03_PRED), "--label-dir", str(LABEL_DIR),
    "--output-dir", str(OUT), "--n120-threshold", str(N120_THRESHOLD),
    "--e03-threshold", str(E03_THRESHOLD),
]
subprocess.run(command, cwd=REPO, check=True)


## 핵심 요약


In [ ]:
summary = json.loads((OUT / "rescue_summary.json").read_text())
display(pd.DataFrame([summary]))
print("E03 net detected-lesion change = rescue − regression =", summary["status_counts"].get("e03_rescue", 0) - summary["status_counts"].get("e03_regression", 0))


## 개별 병변 크기별 rescue / miss


In [ ]:
by_size = pd.read_csv(OUT / "rescue_by_size.csv")
display(by_size)

lesions = pd.read_csv(OUT / "lesion_rescue_detail.csv")
print("E03가 새로 탐지한 병변")
display(lesions[lesions.status == "e03_rescue"].sort_values("gt_volume_ml")[["case_id", "lesion_id", "gt_volume_ml", "size_group"]])
print("E03가 새로 놓친 병변")
display(lesions[lesions.status == "e03_regression"].sort_values("gt_volume_ml")[["case_id", "lesion_id", "gt_volume_ml", "size_group"]])
print("두 모델이 모두 놓친 병변: 가장 작은 20개")
display(lesions[lesions.status == "missed_by_both"].sort_values("gt_volume_ml").head(20)[["case_id", "lesion_id", "gt_volume_ml", "size_group"]])


## Case별 FP 변화


In [ ]:
cases = pd.read_csv(OUT / "case_fp_comparison.csv")
cases["fp_change_e03_minus_n120"] = cases["e03_fp_lesions"] - cases["n120_fp_lesions"]
display(cases.sort_values("fp_change_e03_minus_n120", ascending=False).head(15))
print("평균 FP/scan")
display(cases[["n120_fp_lesions", "e03_fp_lesions"]].mean().to_frame("FP_per_scan"))


## Threshold 안정성


In [ ]:
n120_sweep = pd.read_csv(REPO / "runs/I-nnunet/Dataset502_ISLES2022/nnUNetTrainerIschemiaMultimodal__IschemiaPlansDualEncoderGatedV1__3d_fullres/fold_0/final_metrics/threshold_sweep.csv")
e03_sweep = pd.read_csv(REPO / "runs/ischemia_ensemble/I-E03_N120_S50_S101/N120_060_S101_020_S50_020/ensemble_threshold_sweep.csv")
e03_sweep = e03_sweep[e03_sweep.candidate.str.startswith("soft_vote__")]
for name, data in [("I-N120", n120_sweep), ("E03", e03_sweep)]:
    print(name)
    columns = [x for x in ["threshold", "macro_dice", "macro_dice_positive", "lesion_recall", "lesion_f2", "small_lesion_recall", "fp_per_scan"] if x in data.columns]
    display(data[data.threshold.isin([.15, .20, .25])][columns].sort_values("threshold"))
